# 03 — Prior Distributions
**References:** Gelman et al. BDA3 Ch. 2–3 · Bernardo & Smith (1994) Ch. 5 · Robert (2007) Ch. 3

## Narrative thread
```
Informative -> Non-informative -> Jeffreys -> Reference -> Weakly informative -> Prior predictive checks
```

## 1. Spectrum of priors

| Type | Definition | When to use |
|---|---|---|
| **Informative** | Encodes genuine prior knowledge | Strong domain expertise, small $n$ |
| **Weakly informative** | Regularizes without dominating data | Default recommendation (Gelman) |
| **Non-informative (flat)** | $p(\theta) \propto 1$ | Desire "objective" inference |
| **Jeffreys** | $p(\theta) \propto \sqrt{I(\theta)}$ | Reparametrization invariance |
| **Reference** | Maximizes expected KL divergence | Bernardo's objective prior theory |
| **Improper** | Does not integrate to 1 | Often yields proper posteriors |

## 2. Jeffreys prior

$$p_J(\theta) \propto \sqrt{I(\theta)} = \sqrt{-E\left[\frac{\partial^2}{\partial\theta^2}\log f(X\mid\theta)\right]}$$

**Invariance property:** If $\phi = g(\theta)$, then $p_J(\phi) \propto \sqrt{I_\phi(\phi)}$ — automatically consistent across reparametrizations.

| Model | Jeffreys prior | Notes |
|---|---|---|
| Binomial($p$) | Beta$(1/2, 1/2)$ | Arc-sine distribution |
| Poisson($\lambda$) | $\lambda^{-1/2}$ (improper) | — |
| Normal mean $\mu$ | $\propto 1$ (improper) | Translation invariant |
| Normal $\sigma$ | $\propto 1/\sigma$ (improper) | Scale invariant; $p(\sigma^2)\propto 1/\sigma^2$ |
| Normal $(\mu, \sigma^2)$ | $\propto 1/\sigma^2$ | Do NOT use — leads to problems |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Jeffreys prior: invariance across reparametrizations ──────────────────
# For Binomial(n,p), Jeffreys is Beta(1/2,1/2)
# For reparametrization phi = log(p/(1-p)) (log-odds)
# Jeffreys prior on phi should be the push-forward of Beta(1/2,1/2)

p_grid   = np.linspace(0.001, 0.999, 500)
phi_grid = np.log(p_grid / (1 - p_grid))  # log-odds

jeffreys_p   = stats.beta.pdf(p_grid, 0.5, 0.5)  # Beta(1/2,1/2) in p-space
# Push-forward to phi-space: f_phi(phi) = f_p(p(phi)) * |dp/dphi|
# p(phi) = e^phi/(1+e^phi),  dp/dphi = p*(1-p)
p_of_phi     = np.exp(phi_grid) / (1 + np.exp(phi_grid))
jacobian     = p_of_phi * (1 - p_of_phi)
jeffreys_phi = stats.beta.pdf(p_of_phi, 0.5, 0.5) * jacobian

# Flat prior on p pushed to phi
flat_phi     = 1.0 * jacobian  # flat on p -> U-shaped on phi
flat_phi    /= np.trapz(flat_phi, phi_grid)
jeffreys_phi /= np.trapz(jeffreys_phi, phi_grid)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(p_grid, jeffreys_p, color='#4361ee', lw=2.5, label='Jeffreys = Beta(1/2,1/2)')
axes[0].axhline(1, color='#f72585', lw=2, linestyle='--', label='Flat prior on p')
axes[0].set_xlabel('p'); axes[0].set_ylabel('Density')
axes[0].set_title('Prior on p (probability space)\nJeffreys puts mass near boundaries')
axes[0].legend(fontsize=9)

axes[1].plot(phi_grid, jeffreys_phi, color='#4361ee', lw=2.5, label='Jeffreys on $\\phi$')
axes[1].plot(phi_grid, flat_phi,     color='#f72585', lw=2, linestyle='--', label='Flat-on-p pushed to $\\phi$')
axes[1].set_xlabel('$\\phi = \\log(p/1-p)$'); axes[1].set_ylabel('Density')
axes[1].set_title('Same priors on $\\phi = \\text{logit}(p)$\nJeffreys ≈ flat on $\\phi$; flat-on-p not flat on $\\phi$')
axes[1].set_xlim(-5, 5); axes[1].legend(fontsize=9)

# ── Prior sensitivity: effect of prior choice on posterior ─────────────────
x_obs, n_obs = 3, 20
priors_list = [
    ('Jeffreys Beta(0.5,0.5)', 0.5, 0.5, '#4361ee'),
    ('Uniform Beta(1,1)',       1.0, 1.0, '#06d6a0'),
    ('Skeptical Beta(1,5)',     1.0, 5.0, '#f72585'),
    ('Informative Beta(5,20)', 5.0, 20.0, '#ff9f1c'),
]
ax3 = axes[2]
for name, a0, b0, c in priors_list:
    ap, bp = a0 + x_obs, b0 + n_obs - x_obs
    ax3.plot(p_grid, stats.beta.pdf(p_grid, ap, bp), lw=2.5, color=c,
             label=f'{name}\nPost. mean={ap/(ap+bp):.3f}')
ax3.axvline(x_obs/n_obs, color='k', lw=1.5, linestyle=':', label=f'MLE={x_obs/n_obs}')
ax3.set_xlabel('p'); ax3.set_title(f'Prior sensitivity: x={x_obs}, n={n_obs}\nSmall n → prior matters more')
ax3.legend(fontsize=7)

plt.suptitle('Jeffreys Prior: Reparametrization Invariance & Prior Sensitivity', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Weakly informative priors (Gelman's recommendation)

The goal is to be **regularizing** without being **informative** — excluding implausible parameter values while letting the data speak for parameters in the plausible range.

| Parameter type | Recommended weakly informative prior |
|---|---|
| Location (unbounded) | $\mathcal{N}(0, \sigma^2)$ where $\sigma$ covers the plausible range |
| Scale (positive) | Half-Normal, Half-Cauchy, Exponential |
| Probability | Beta$(1,1)$ (uniform) or Beta$(2,2)$ (gentle hill) |
| Correlation | LKJ prior with $\eta > 1$ |
| Regression coefs | $\mathcal{N}(0, 1)$ after standardizing predictors |

**Prior predictive check:** Before fitting, simulate data from the prior:
$$\tilde{\mathbf{x}} \sim \int p(\tilde{\mathbf{x}} \mid \theta)\, p(\theta)\, d\theta$$
If simulated data looks implausible, the prior needs revision.

In [ ]:
# ── Prior predictive checks: weakly informative vs diffuse ────────────────
# Logistic regression: y ~ Bernoulli(logistic(alpha + beta*x))
# Prior 1 (diffuse):  alpha,beta ~ N(0, 10^2)
# Prior 2 (weakly informative): alpha,beta ~ N(0, 1)

n_prior_samples = 200
n_x = 50
x   = np.linspace(-3, 3, n_x)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for row, (prior_sd, prior_name) in enumerate([(10, 'Diffuse N(0,100)'), (1, 'Weakly informative N(0,1)')]):
    # Prior predictive simulations
    for i in range(n_prior_samples):
        alpha = np.random.normal(0, prior_sd)
        beta  = np.random.normal(0, prior_sd)
        p_x   = 1 / (1 + np.exp(-(alpha + beta * x)))
        axes[row, 0].plot(x, p_x, lw=0.5, alpha=0.15, color='#4361ee')

    axes[row, 0].set_xlabel('x'); axes[row, 0].set_ylabel('P(y=1|x)')
    axes[row, 0].set_title(f'{prior_name}\nPrior predictive: P(y=1|x)')
    axes[row, 0].set_ylim(0, 1)

    # Distribution of prior-predicted success rates
    prior_rates = []
    for _ in range(2000):
        alpha = np.random.normal(0, prior_sd)
        beta  = np.random.normal(0, prior_sd)
        p_x   = 1 / (1 + np.exp(-(alpha + beta * x)))
        prior_rates.append(p_x.mean())

    axes[row, 1].hist(prior_rates, bins=50, density=True, color='#4361ee', alpha=0.7, edgecolor='white')
    axes[row, 1].set_xlabel('Prior-predicted success rate'); axes[row, 1].set_ylabel('Density')
    axes[row, 1].set_title(f'{prior_name}\nDistribution of prior-predicted $\\bar p$')

plt.suptitle('Prior Predictive Checks: Diffuse vs Weakly Informative Priors\n'
             'Diffuse prior implies extreme probabilities near 0 and 1 — often implausible',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Key takeaways

| Prior type | Key property | Pitfall |
|---|---|---|
| Flat | Simple, "objective" | Not invariant; can be improper and unintuitive |
| Jeffreys | Reparametrization invariant | Can be improper; wrong for multi-parameter |
| Informative | Uses expert knowledge | Hard to elicit; results non-reproducible |
| Weakly informative | Regularizes gently | Requires thought about scale |

**Golden rule (Gelman):** Always do a **prior predictive check** — simulate data from the prior and ask: "Does this look like data I could plausibly observe?"

**Next:** notebook 04 — posterior computation: when analytical posteriors don't exist.